In [5]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///../data/powerlifting.db')

# Shape
pd.read_sql("SELECT COUNT(*) as total_rows FROM meets", engine)

,total_rows
0,3907789


In [6]:
# Equipment Types
pd.read_sql("""
SELECT equipment, COUNT(*) as count
FROM meets
GROUP BY equipment
ORDER BY count DESC
""", engine)

,Equipment,count
0,Raw,1831773
1,Single-ply,1386621
2,Unlimited,323350
3,Wraps,235624
4,Multi-ply,130305
5,Straps,116


In [7]:
# Completeness of key columns
pd.read_sql("""
SELECT
    COUNT(*) as total,
    COUNT(Age) as has_age,
    COUNT(BodyweightKg) as has_bodyweight,
    COUNT(TotalKg) as has_total
FROM meets
""", engine)

,total,has_age,has_bodyweight,has_total
0,3907789,2456780,3863576,3642268


In [8]:
# Sex Breakdown
pd.read_sql("""
SELECT Sex, COUNT(*) as count
FROM meets
GROUP BY Sex
""", engine)

,Sex,count
0,F,1090584
1,M,2817039
2,Mx,166


# Initial Data Cleaning Decisions
Raw is the largest category with 1.8M rows. For now, I'll filter to RAW only

Age - 63% of the entries have age recorded. This is the most significant issue with missing data, and we'll solve it by filtering the rows with missing ages out for now. 

Total - Missing total's are negligible.

Sex - 166 Mx entries are a negligible part of the data set. I'll filter them out.

Mention these decisions in README.

In [9]:
# Check Place column - filter out DQs
pd.read_sql("""
SELECT Place, COUNT(*) as count
FROM meets
WHERE Equipment = 'Raw'
GROUP BY Place
ORDER BY count DESC
LIMIT 15
""", engine)

,Place,count
0,1,912481
1,2,298524
2,3,160506
3,4,98378
4,DQ,66053
5,5,65673
6,6,45995
7,7,33330
8,8,24895
9,9,18957


In [10]:
# Check Event types - want full SBD
pd.read_sql("""
SELECT Event, COUNT(*) as count
FROM meets
WHERE Equipment = 'Raw'
GROUP BY Event
ORDER BY count DESC
""", engine)

,Event,count
0,SBD,1122435
1,B,479308
2,D,158879
3,BD,60846
4,S,9273
5,SD,585
6,SB,447


In [13]:
# Age distribution after filtering to Raw, valid total, valid age
pd.read_sql("""
SELECT
    MIN(Age) as min_age,
    MAX(Age) as max_age,
    AVG(Age) as avg_age
FROM meets
WHERE Equipment = 'Raw'
    AND TotalKg IS NOT NULL
    AND Age IS NOT NULL
""", engine)

,min_age,max_age,avg_age
0,0.0,105.5,30.702931


# Further Data Cleaning Decisions
Filter out "DQ" and "G" to get numeric places only (for completed, valid lifts)

Filter event to SBD only

Min and max ages at 0.0 and 105.5 are unrealistic and data quality issues. Average age is 30.7 which seems reasonable (though maybe slightly inflated?). Age cap to something like 14-80. 